# pile-scan: make a test pile with an answer key

Turns a born-digital PDF into a pile of page images that look like a scanned dump, and keeps the truth in a separate answer key, so the scanner (notebook 1) and the sort pass (notebook 2) can be scored.

By default it uses **United States Reports, volume 586** (the Supreme Court's October Term 2018, public domain, from govinfo.gov):

- Every page is rendered to a grayscale PNG, so the PDF's text layer is gone.
- A share of pages is roughed up to look photocopied (grey paper, slight skew, speckle, edge shadow, blur, JPEG artefacts).
- A share of pages is turned 90, 180 or 270 degrees, like a portrait page saved as landscape.
- The order is partly shuffled: most pages keep their neighbours, a few groups of pages move elsewhere, and a few single pages land far away.
- The table of cases and the index are taken out of the pile and kept in the answer key, to compare with what notebook 2 rebuilds.
- Each page's text from the PDF's text layer goes in the answer key as the ground truth for transcription.

**No GPU needed.** Use a CPU runtime (Runtime > Change runtime type > CPU). The default slice of 145 pages takes about a minute; the whole volume a few minutes more, mostly writing to Drive.

Then run notebook 1 with `INPUT_DIR` set to the `pages` folder this prints. Never point it at the corpus folder itself: it would read the answer key too.

In [ ]:
#@title Settings { display-mode: "form" }
#@markdown **Where things go on your Drive.** The corpus folder gets `pages/` (the pile) and `answer_key/`.
CORPUS_DIR = "/content/drive/MyDrive/pile-scan/testpiles/us586"  #@param {type:"string"}
#@markdown The source PDF is downloaded here once, then reused.
SOURCE_PDF = "/content/drive/MyDrive/pile-scan/sources/USREPORTS-586.pdf"  #@param {type:"string"}
SOURCE_URL = "https://www.govinfo.gov/content/pkg/USREPORTS-586/pdf/USREPORTS-586.pdf"  #@param {type:"string"}

#@markdown **Which pages.** Printed page numbers ("1-145" is the first eleven opinions, each complete), `all` for the whole volume, or PDF pages like `pdf:201-345`.
PAGES = "1-145"  #@param {type:"string"}
#@markdown Sections kept out of the pile and saved in the answer key: cover, front_matter, table_of_cases, opinions, orders, index, other.
HOLD_OUT = "cover, table_of_cases, index"  #@param {type:"string"}

#@markdown **How rough.** Shares are fractions of the pages in the pile.
ROTATE_SHARE = 0.1  #@param {type:"slider", min:0, max:1, step:0.05}
DEGRADE_SHARE = 0.5  #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown `partial`: mostly in order, some groups and single pages moved. `none`: true order. `full`: random order.
SHUFFLE = "partial"  #@param ["partial", "none", "full"]
#@markdown `pile`: scan_00001.png, scan_00002.png ... in pile order, like a scanner's numbering. `random`: random names, so the scanner sees the pages in an order unrelated to the pile.
FILENAMES = "pile"  #@param ["pile", "random"]
SEED = 586  #@param {type:"integer"}
DPI = 200  #@param {type:"integer"}
#@markdown Replace an existing corpus in CORPUS_DIR.
OVERWRITE = False  #@param {type:"boolean"}

#@markdown **Code to install**
REPO = "daverooney/pile-scan"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Install (uses uv and the repo's lockfile)
# If the repo is private, add a Colab secret named GITHUB_TOKEN (key icon on the left)
# holding a read-only GitHub token, and give this notebook access to it.
import shutil, subprocess, sys

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

_token = _secret("GITHUB_TOKEN")
_auth = f"{_token}@" if _token else ""
CODE_DIR = "/content/pile-scan"
shutil.rmtree(CODE_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                f"https://{_auth}github.com/{REPO}.git", CODE_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# Pinned versions from uv.lock, installed into Colab's own Python.
subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", f"{CODE_DIR}/requirements-colab.txt"], check=True)
subprocess.run(["uv", "pip", "install", "--system", "-q", "--no-deps", "--reinstall", CODE_DIR], check=True)

import pile_scan
print("pile-scan", pile_scan.__version__)

In [ ]:
#@title Make the pile and the answer key
from pile_scan import testcorpus

cfg = testcorpus.CorpusConfig(
    corpus_dir=CORPUS_DIR, source_pdf=SOURCE_PDF, source_url=SOURCE_URL.strip() or None,
    pages=PAGES, hold_out=testcorpus.parse_hold_out(HOLD_OUT), seed=int(SEED), dpi=int(DPI),
    rotate_share=float(ROTATE_SHARE), degrade_share=float(DEGRADE_SHARE),
    shuffle=SHUFFLE, filenames=FILENAMES, overwrite=OVERWRITE,
)
summary = testcorpus.make_corpus(cfg)
print()
print("For notebook 1, set INPUT_DIR to:")
print(f'    "{summary.pages_dir}"')

In [ ]:
#@title Look at a few pages and the answer key
import pandas as pd
from PIL import Image
from IPython.display import display

key = pd.read_csv(summary.answer_dir / "pages.csv")
show = pd.concat([key[key.rotation_cw != 0].head(2), key[key.degraded].head(2), key[key.moved.notna()].head(2)])
for _, row in show.drop_duplicates("file").iterrows():
    print(f"{row.file}: printed page {row.printed_page}, turned {row.rotation_cw} degrees clockwise, "
          f"{'roughed up' if row.degraded else 'clean'}{', moved: ' + row.moved if isinstance(row.moved, str) else ''}")
    im = Image.open(summary.pages_dir / row.file)
    im.thumbnail((700, 700))
    display(im)
key.drop(columns=["width", "height"]).head(30)

## What's in the answer key

- `pages.csv` / `pages.jsonl`: one row per image in pile order, with its true position, PDF page, printed page number, section, the opinion it belongs to (`document`, `title`), the rotation and photocopy settings applied, and whether it was moved in a group or alone. The `.jsonl` adds the page's text from the PDF's text layer.
- `true_order.txt`: the image names in true reading order.
- `held_out/`: the table of cases and the index, as text and as page images.
- `manifest.json`: settings, the source PDF's SHA-256, and where each section starts and ends in the PDF.

One caveat for scoring transcriptions: govinfo's text of this volume drops the "fi", "fl" and "ffi" ligatures ("defnition", "offce", "fled" for "filed"), and the PDF's text layer most likely does too, so a correct reading of those words can look like a mismatch. The first run will show: look for "defnition" in `pages.jsonl`.